## 1) Environment Setup

In [ ]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
project_root = Path.cwd()
script_dir = project_root / "Individual Activity"

if IN_COLAB and not script_dir.exists():
    repo_url = "https://github.com/LorenzoBela/Machine-Learning-Group7.git"
    target_dir = Path.cwd() / "Machine-Learning-Group7"
    if not target_dir.exists():
        subprocess.run(["git", "clone", repo_url], check=True)
    project_root = target_dir
    script_dir = project_root / "Individual Activity"

if not script_dir.exists():
    raise FileNotFoundError(
        f"Could not find Individual Activity folder from: {project_root}"
    )

print(f"Running in Colab: {IN_COLAB}")
print(f"Project root: {project_root}")
print(f"Script folder: {script_dir}")

## 2) Import the Existing Runner

In [ ]:
import importlib

if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

import run_activity_rounds as rar
rar = importlib.reload(rar)

print(f"Loaded module: {rar.__file__}")

## 3) Configure and Run

In [ ]:
from tqdm.auto import tqdm as tqdm_auto

rar.tqdm = tqdm_auto
if hasattr(rar, "is_notebook_runtime"):
    rar.is_notebook_runtime = lambda: True

worksheet_md = script_dir / "Individual Activity.md"

if not worksheet_md.exists():
    required_headers = [
        "## Round 1 - Baseline",
        "## Round 2 - Learning Rate",
        "## Round 3 - More Data and More Epochs",
        "## Round 4 - Fine-Tuning (unfreeze the backbone)",
        "## Round 5 - Data Augmentation",
        "## Round 6 - Learning Rate Scheduler",
        "## Round 7 - Your Best Configuration",
    ]
    worksheet_md.write_text("\n\n".join(required_headers) + "\n", encoding="utf-8")

RUN_ARGS = [
    "--skip-kaggle",
    "--seed", "42",
    "--worksheet-md", str(worksheet_md),
]

argv_backup = sys.argv[:]
sys.argv = ["run_activity_rounds.py", *RUN_ARGS]
try:
    rar.main()
finally:
    sys.argv = argv_backup

## 4) Quick Result Check

In [ ]:
import json
import pandas as pd
from IPython.display import display

results_json = script_dir / "activity_results.json"
results_csv = script_dir / "activity_results_flat.csv"

with results_json.open("r", encoding="utf-8") as f:
    payload = json.load(f)

print(f"Round 1 test accuracy: {payload['round1_test_acc'] * 100:.2f}%")
print(f"Round 7 test accuracy: {payload['round7']['test_acc'] * 100:.2f}%")
print(f"Improvement: {payload['improvement_over_round1_pp']:.2f} percentage points")
print(f"Artifacts dir: {payload['meta']['artifacts_dir']}")

df = pd.read_csv(results_csv)
display(df[["run_id", "test_acc", "final_train_acc", "final_val_acc", "train_val_gap"]])